# Run Data100 PFG_MOEA_D Change Population Size

Notebook nay chay rieng `PFG_MOEA_D` cho data100 voi `POP_SIZE = 100, 200, 300, 400, 500`. Thoi gian chay cua moi instance duoc lay tu file `meta_info.csv` trong root cu `result_instance_5algo_with_stop_time_56`, tuc ket qua POP_SIZE=100 da co san.

Ket qua duoc luu vao `result_instance_data100_PFG_MOEA_D_change_pop/pop_100`, `pop_200`, `pop_300`, `pop_400`, `pop_500`. Voi `pop_100`, notebook copy ket qua cu da co san thay vi chay lai.


In [1]:
import os
import sys
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

PROJECT_DIR = Path(r"D:\codePython\pythonProject\final_project\pfg_moead_vrpd_ver2")
os.chdir(PROJECT_DIR)
for path in [PROJECT_DIR.parent, PROJECT_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from pfg_moead_vrpd_ver2.algorithm.pfg_moead_solver_stop import PFGMOEADSolverStop
from pfg_moead_vrpd_ver2.model.customer import Customer

# CONFIG
POP_SIZES = [100, 200, 300, 400, 500]
NUM_TRUCKS = 6
SEED = 1
ALGO = "PFG_MOEA_D"

DATA_DIR = Path("data")
SOURCE_RESULT_ROOT = Path("result_instance_5algo_with_stop_time_56")
RESULT_ROOT = Path("result_instance_data100_PFG_MOEA_D_change_pop")
RESULT_ROOT.mkdir(exist_ok=True)

# False = neu ket qua cua instance/pop_size da ton tai thi bo qua, khong chay lai.
# True = bat buoc chay lai va ghi de ket qua cu.
FORCE_RERUN = False


In [2]:
# LOAD DATASET + META HELPERS

def load_customers_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    customers = {}

    depot = df.iloc[0]
    customers[0] = Customer(
        cid=0,
        x=float(depot["x"]),
        y=float(depot["y"]),
        demand=0.0,
        ready_time=float(depot["open"]),
        due_time=float(depot["close"]),
        service_time=float(depot["servicetime"]),
        drone_serve=False,
        time=0.0,
    )

    for idx in range(1, len(df)):
        r = df.iloc[idx]
        customers[idx] = Customer(
            cid=idx,
            x=float(r["x"]),
            y=float(r["y"]),
            demand=float(r["demand"]),
            ready_time=float(r["open"]),
            due_time=float(r["close"]),
            service_time=float(r["servicetime"]),
            drone_serve=bool(r["drone_serve"]),
            time=float(r["time"]),
        )

    return customers


def read_source_max_time(instance_name):
    meta_file = SOURCE_RESULT_ROOT / f"result_{instance_name}" / "meta_info.csv"
    if not meta_file.exists():
        raise FileNotFoundError(f"Missing source meta_info.csv: {meta_file}")

    meta = pd.read_csv(meta_file)
    row = meta[meta["algorithm"] == ALGO]
    if row.empty:
        raise ValueError(f"Algorithm {ALGO} not found in {meta_file}")

    return float(row.iloc[0]["max_time"]), meta_file


def output_dir_for(pop_size, instance_name):
    return RESULT_ROOT / f"pop_{pop_size}" / f"result_{instance_name}"


In [3]:
# SAVE RESULT HELPERS

def print_and_save_pareto(pf, algo_name, result_dir):
    print("\n" + "=" * 80)
    print(f"{algo_name} Pareto Front")
    print(f"Pareto size = {len(pf)}")
    print("=" * 80)

    rows = []

    for i, s in enumerate(pf, 1):
        print(f"\n--- Solution {i} ---")
        print(f"Makespan = {s.makespan:.4f}")
        print(f"Carbon   = {s.carbonEmission:.4f}")

        for t, route in enumerate(s.truckRoutes):
            drone_list = []
            if t < len(s.droneCustomers):
                drone_list = sorted(list(s.droneCustomers[t]))

            print(f"Truck {t + 1}: {route}")
            print(f"Drone {t + 1}: {drone_list}")

            rows.append({
                "algorithm": algo_name,
                "solution_id": i,
                "truck_id": t + 1,
                "makespan": s.makespan,
                "carbon": s.carbonEmission,
                "truck_route": " ".join(map(str, route)),
                "drone_customers": " ".join(map(str, drone_list)),
            })

    df = pd.DataFrame(rows)
    csv_path = result_dir / f"pareto_{algo_name}.csv"
    df.to_csv(csv_path, index=False)
    print("\nSaved Pareto tours to:", csv_path)


def save_meta_info(result_dir, pop_size, max_time, run_time, generations, source_meta_file):
    df_meta = pd.DataFrame([
        {
            "algorithm": ALGO,
            "pop_size": pop_size,
            "max_time": max_time,
            "run_time": run_time,
            "generations": generations,
            "source_meta_file": str(source_meta_file),
        }
    ])

    meta_path = result_dir / "meta_info.csv"
    df_meta.to_csv(meta_path, index=False)
    print("Saved meta info to:", meta_path)


def copy_pop100_from_source(instance_name):
    source_dir = SOURCE_RESULT_ROOT / f"result_{instance_name}"
    result_dir = output_dir_for(100, instance_name)
    result_dir.mkdir(parents=True, exist_ok=True)

    source_pareto = source_dir / f"pareto_{ALGO}.csv"
    source_meta = source_dir / "meta_info.csv"
    target_pareto = result_dir / f"pareto_{ALGO}.csv"
    target_meta = result_dir / "meta_info.csv"

    if target_pareto.exists() and target_meta.exists() and not FORCE_RERUN:
        print("POP_SIZE=100 result already exists, skip copy:", result_dir)
        return

    if not source_pareto.exists():
        raise FileNotFoundError(f"Missing source Pareto CSV: {source_pareto}")
    if not source_meta.exists():
        raise FileNotFoundError(f"Missing source meta_info.csv: {source_meta}")

    shutil.copy2(source_pareto, target_pareto)

    meta = pd.read_csv(source_meta)
    meta = meta[meta["algorithm"] == ALGO].copy()
    if meta.empty:
        raise ValueError(f"Algorithm {ALGO} not found in {source_meta}")

    meta["pop_size"] = 100
    meta["source_meta_file"] = str(source_meta)
    meta.to_csv(target_meta, index=False)

    print("Copied POP_SIZE=100 Pareto to:", target_pareto)
    print("Copied POP_SIZE=100 meta to:", target_meta)


In [4]:
# RUN ONE INSTANCE FOR ONE POP SIZE

def run_instance_with_pop(instance_name, customers, pop_size):
    result_dir = output_dir_for(pop_size, instance_name)
    pareto_file = result_dir / f"pareto_{ALGO}.csv"
    meta_file = result_dir / "meta_info.csv"

    print(f"\n========== INSTANCE {instance_name} | POP_SIZE={pop_size} ==========")

    if pop_size == 100:
        copy_pop100_from_source(instance_name)
        return

    if pareto_file.exists() and meta_file.exists() and not FORCE_RERUN:
        print("Result already exists, skip:", result_dir)
        return

    result_dir.mkdir(parents=True, exist_ok=True)

    max_time, source_meta_file = read_source_max_time(instance_name)
    print("Source meta:", source_meta_file)
    print("Use max_time:", max_time)

    random.seed(SEED)
    np.random.seed(SEED)

    solver = PFGMOEADSolverStop(
        pop_size=pop_size,
        max_time=max_time,
        num_trucks=NUM_TRUCKS,
        customers=customers,
        seed=SEED,
    )

    start = time.time()
    pf = solver.run()
    run_time = time.time() - start

    print_and_save_pareto(pf, ALGO, result_dir)
    save_meta_info(
        result_dir=result_dir,
        pop_size=pop_size,
        max_time=max_time,
        run_time=run_time,
        generations=getattr(solver, "generations", None),
        source_meta_file=source_meta_file,
    )


In [5]:
# MAIN LOOP
INSTANCE_FILES = sorted([
    f for f in DATA_DIR.iterdir()
    if f.is_file() and f.suffix.lower() == ".csv"
])

print("Total instances:", len(INSTANCE_FILES))
print("POP sizes:", POP_SIZES)

for file_path in tqdm(INSTANCE_FILES):
    instance_name = file_path.stem
    customers = load_customers_from_csv(file_path)

    for pop_size in POP_SIZES:
        run_instance_with_pop(instance_name, customers, pop_size)


Total instances: 56
POP sizes: [100, 200, 300, 400, 500]


  0%|          | 0/56 [00:00<?, ?it/s]


========== INSTANCE h100c101 | POP_SIZE=100 ==========
Copied POP_SIZE=100 Pareto to: result_instance_data100_PFG_MOEA_D_change_pop\pop_100\result_h100c101\pareto_PFG_MOEA_D.csv
Copied POP_SIZE=100 meta to: result_instance_data100_PFG_MOEA_D_change_pop\pop_100\result_h100c101\meta_info.csv

========== INSTANCE h100c101 | POP_SIZE=200 ==========
Source meta: result_instance_5algo_with_stop_time_56\result_h100c101\meta_info.csv
Use max_time: 556.1338531970978


  0%|          | 0/56 [01:50<?, ?it/s]


KeyboardInterrupt: 